> 📄 English: [agent_memory_tutorial.ipynb](agent_memory_tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial)

# 使用 Redis 的 Agent 记忆

## 🚀 在 Google Colab 中运行

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-memory-with-redis/agent_memory_tutorial.ipynb)


## 简介

没有记忆的 AI agent 就像金鱼——每次对话结束后就忘记一切，无法从过往交互中学习，也无法跨会话保持上下文。Agentic 系统需要**短期**和**长期**记忆，才能以个性化、稳健的方式完成任务。记忆本质上就是状态管理，而 [**Redis**](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=redis-try-free&target=https%3A%2F%2Fredis.io%2Ftry-free%2F%3Futm_source%3Dnir%26utm_medium%3Dcpa%26utm_campaign%3D2025-05-ai_in_production-influencer-nir%26utm_content%3Dsd-software_download-7013z000001WaRY&text=%2A%2ARedis%2A%2A) 正是当今生产系统中用于这类场景的知名内存数据库。

## 我们要构建什么

本教程演示如何使用 **Redis** 和 **LangGraph** 构建一个**具备记忆的旅行 agent**，它能记住用户偏好并提供个性化推荐。这是一个**横向通用概念**，你可以把它应用到自己的 agent 用例中。

我们将探索：

1. 使用 LangGraph 的 checkpointer 管理短期记忆
2. 使用 RedisVL 存储和检索长期记忆
3. 把长期记忆作为 ReAct agent 的工具来管理
4. 通过摘要（summarization）管理对话历史的大小

# 🧠 记忆架构概览

我们的 agent 使用双记忆系统：
- **短期**：管理对话上下文
- **长期**：存储持久化知识

## 短期记忆
agent 通过 LangGraph 的 [checkpointer](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=github-langgraph-redis&target=https%3A%2F%2Fgithub.com%2Fredis-developer%2Flanggraph-redis&text=checkpointer) 使用 Redis 跟踪聊天历史。图中的每个节点（Retrieve Memories、Respond、Summarize）都会把状态保存到 Redis，包括对话历史和 thread 元数据。

<img src="assets/short-term-memory.png" width="400" alt="Short-term Memory Diagram">

为了防止上下文窗口被污染，当对话超过可配置长度时，agent 会对其进行摘要。

## 长期记忆

长期记忆使用 RedisVL 客户端在 Redis 中存储和索引，分为两种类型：
- **Episodic（情景记忆）**：用户偏好和经历
- **Semantic（语义记忆）**：通用旅行知识

<img src="assets/long-term-memory.png" width="600" alt="Long-term Memory Diagram">

>**注意**：这些记忆类型与 [CoALA](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=arxiv-2309-02427&target=https%3A%2F%2Farxiv.org%2Fabs%2F2309.02427&text=CoALA) 论文中的概念一致。我们 agent 的程序性记忆（procedural memory）编码在它的 Python 工作流中。

---

# 配置我们的环境

在深入代码之前，让我们用正确的 Python 库来配置开发环境。

>**注意**：安装完库之后，你可能需要重启 kernel。

In [ ]:
%pip install langchain-openai langgraph-checkpoint langgraph langgraph-checkpoint-redis langchain-redis

## 所需的 API 密钥

本教程需要你添加一个带计费信息的 [OpenAI API](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fsignup&text=OpenAI%20API) 密钥。

In [ ]:
import getpass
import os

def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}:")


_set_env("OPENAI_API_KEY")

## 配置 Redis

运行 Redis 有两种方式：

1. **Redis Cloud**：如需全托管、无缝的体验，请使用 [Redis Cloud 的免费实例](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=redis-try-free&target=https%3A%2F%2Fredis.io%2Ftry-free%2F%3Futm_source%3Dnir%26utm_medium%3Dcpa%26utm_campaign%3D2025-05-ai_in_production-influencer-nir%26utm_content%3Dsd-software_download-7013z000001WaRY&text=a%20free%20instance%20of%20Redis%20Cloud)。
2. **本地 Redis**：如需简单、本地（非持久化）的 Redis 实例，请运行下面的单元格。

运行下面的单元格，在你的 Google Colab 服务器上启动一个本地 Redis 实例。

In [ ]:
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

让我们测试一下 Redis 连接，并创建一个客户端与服务器通信。

In [ ]:
import os

from redis import Redis

# Use the environment variable if set, otherwise default to localhost
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()

## 准备记忆数据模型

在本节中，我们将使用 `Pydantic` 为 agent 的记忆创建一个健壮的数据建模系统。这些模型将确保类型安全，并为在 Redis 中存储和检索记忆提供清晰的数据结构。

我们将实现四个关键组件：

1. `MemoryType` - 一个枚举，将记忆分为两种类型：
   - Episodic（情景记忆）：个人经历和用户偏好
   - Semantic（语义记忆）：通用知识和领域事实

2. `Memory` - 核心模型，表示单条记忆条目及其内容和元数据

3. `Memories` - 容器模型，用于保存记忆对象的集合

4. `StoredMemory` - 针对已持久化到 Redis 的记忆的特化模型

这些模型协同工作，形成一个完整的记忆生命周期：从创建到存储再到检索。

In [ ]:
import ulid

from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field


class MemoryType(str, Enum):
    """
    Defines the type of long-term memory for categorization and retrieval.

    EPISODIC: Personal experiences and user-specific preferences
              (e.g., "User prefers Delta airlines", "User visited Paris last year")

    SEMANTIC: General domain knowledge and facts
              (e.g., "Singapore requires passport", "Tokyo has excellent public transit")

    The type of a long-term memory.

    EPISODIC: User specific experiences and preferences

    SEMANTIC: General knowledge on top of the user's preferences and LLM's
    training data.
    """

    EPISODIC = "episodic"
    SEMANTIC = "semantic"


class Memory(BaseModel):
    """Represents a single long-term memory."""

    content: str
    memory_type: MemoryType
    metadata: str


class Memories(BaseModel):
    """
    A list of memories extracted from a conversation by an LLM.

    NOTE: OpenAI's structured output requires us to wrap the list in an object.
    """

    memories: List[Memory]


class StoredMemory(Memory):
    """A stored long-term memory"""

    id: str  # The redis key
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None

现在，我们有了类型安全的数据模型，可以处理从 LLM 提取到 Redis 存储的完整记忆生命周期，并为生产使用提供合适的元数据跟踪。接下来，我们将配置 Redis 基础设施，使用向量嵌入来存储和搜索这些记忆。

# 记忆存储

- **短期记忆**由 `langgraph-checkpoint-redis` 中的 `RedisSaver` 自动处理。
- 对于**长期记忆**，我们将使用 RedisVL 配合向量嵌入，对过往经历和知识进行语义搜索。

下面，我们将在 Redis 中创建一个搜索索引 schema 来存放长期记忆。这个 schema 包含若干字段，包括内容、记忆类型、元数据、时间戳、用户 id、记忆 id，以及记忆的嵌入向量。

In [ ]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema


# Define the schema for our vector search index
# This creates the structure for storing and querying memories
memory_schema = IndexSchema.from_dict({
        "index": {
            "name": "agent_memories",  # Index name for identification
            "prefix": "memory",       # Redis key prefix (memory:1, memory:2, etc.)
            "key_separator": ":",
            "storage_type": "json",
        },
        "fields": [
            {"name": "content", "type": "text"},
            {"name": "memory_type", "type": "tag"},
            {"name": "metadata", "type": "text"},
            {"name": "created_at", "type": "text"},
            {"name": "user_id", "type": "tag"},
            {"name": "memory_id", "type": "tag"},
            {
                "name": "embedding",
                "type": "vector",
                "attrs": {
                    "algorithm": "flat",
                    "dims": 1536,  # OpenAI embedding dimension
                    "distance_metric": "cosine",
                    "datatype": "float32",
                },
            },
        ],
    }
)

下面我们根据 `IndexSchema` 和 Redis 客户端连接对象创建 `SearchIndex`。如果索引 spec 已经存在，我们会将其覆盖！

In [ ]:
try:
    long_term_memory_index = SearchIndex(
        schema=memory_schema,
        redis_client=redis_client,
        validate_on_load=True
    )
    long_term_memory_index.create(overwrite=True)
    print("Long-term memory index ready")
except Exception as e:
    print(f"Error creating index: {e}")

现在索引已创建，我们可以使用 `rvl` 命令行工具检查 Redis 中的长期记忆索引：

In [ ]:
!rvl index info -i agent_memories

## 访问记忆的函数

接下来，我们提供三个核心函数来访问、存储和检索记忆。最终我们会在供 LLM 调用的工具中使用它们。我们先加载一个 vectorizer 类来创建 OpenAI 嵌入。

In [ ]:
from redisvl.utils.vectorize.text.openai import OpenAITextVectorizer

openai_embed = OpenAITextVectorizer(model="text-embedding-ada-002")

接下来，我们设置一个简单的 logger，让我们的函数记录正在发生的事情的日志活动。

In [ ]:
import logging

# Set up a logger
logger = logging.getLogger(__name__)

### 1. 检查相似记忆
首先，我们编写一个工具函数，检查索引中是否已经存在与给定记忆相似的记忆。

该函数通过以下方式检查 Redis 中的重复记忆：
1. 将输入内容转换为向量嵌入
2. 为 user_id 和 memory_type 创建过滤器
3. 使用向量范围查询进行向量相似度搜索，查找任何已存在且相似的记忆
4. 如果存在相似记忆则返回 True，否则返回 False

这有助于避免在 agent 的记忆中存储冗余信息。

In [ ]:
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag


# If we have any memories that aren't associated with a user, we'll use this ID.
SYSTEM_USER_ID = "system"


def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    """Check if a similar long-term memory already exists in Redis."""
    content_embedding = openai_embed.embed(content)

    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type)

    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)

    # Search for similar memories
    vector_query = VectorRangeQuery(
        vector=content_embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    results = long_term_memory_index.query(vector_query)
    logger.debug(f"Similar memory search results: {results}")

    if results:
        logger.debug(
            f"{len(results)} similar {'memory' if results.count == 1 else 'memories'} found. First: "
            f"{results[0]['id']}. Skipping storage."
        )
        return True

    return False

### 2. 存储长期记忆

下面是一个负责在 Redis 中存储长期记忆、并内置去重的函数。

它是我们记忆系统的关键部分，能够：
1. 通过检查相似内容防止重复记忆
2. 创建向量嵌入以支持语义搜索
3. 存储记忆及其相关元数据，便于将来检索

存储记忆时，我们会使用 `similar_memory_exists()` 函数进行内联记忆去重。

In [ ]:


from datetime import datetime
from typing import List, Optional, Union

import ulid


def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
):
    """Store a long-term memory in Redis with deduplication.

        This function:
        1. Checks for similar existing memories to avoid duplicates
        2. Generates vector embeddings for semantic search
        3. Stores the memory with metadata for retrieval
        """
    if metadata is None:
        metadata = "{}"

    logger.info(f"Preparing to store memory: {content}")

    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory found, skipping storage")
        return

    embedding = openai_embed.embed(content)

    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id,
    }

    try:
        long_term_memory_index.load([memory_data])
    except Exception as e:
        logger.error(f"Error storing memory: {e}")
        return

    logger.info(f"Stored {memory_type} memory: {content}")

### 3. 检索相关的长期记忆
既然我们已经在存储记忆，现在可以使用 RedisVL 通过带元数据过滤的向量相似度搜索来检索它们。

该函数：
1. 接收一个查询字符串、可选的过滤器（记忆类型、用户 ID、thread ID）以及一个距离阈值（语义）
2. 使用查询的嵌入创建向量范围查询
3. 根据传入的选项构建过滤器对象
4. 过滤以缩小搜索结果范围
5. 执行搜索并返回解析后的记忆对象

In [ ]:
def retrieve_memories(
    query: str,
    memory_type: Union[Optional[MemoryType], List[MemoryType]] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
    limit: int = 5,
) -> List[StoredMemory]:
    """Retrieve relevant memories from Redis using vector similarity search.

    """
    # Create vector query using query embedding
    logger.debug(f"Retrieving memories for query: {query}")
    vector_query = VectorRangeQuery(
        vector=openai_embed.embed(query),
        return_fields=[
            "content",
            "memory_type", 
            "metadata",
            "created_at",
            "memory_id",
            "thread_id",
            "user_id",
        ],
        num_results=limit,
        vector_field_name="embedding",
        dialect=2,
        distance_threshold=distance_threshold,
    )

    # Build filter conditions
    base_filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]

    if memory_type:
        if isinstance(memory_type, list):
            base_filters.append(f"@memory_type:{{{'|'.join(memory_type)}}}")
        else:
            base_filters.append(f"@memory_type:{{{memory_type.value}}}")

    if thread_id:
        base_filters.append(f"@thread_id:{{{thread_id}}}")

    vector_query.set_filter(" ".join(base_filters))

    # Execute vector similarity search
    results = long_term_memory_index.query(vector_query)

    # Parse results into StoredMemory objects
    memories = []
    for doc in results:
        try:
            memory = StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id", None),
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            )
            memories.append(memory)
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
            continue
    return memories

## 🛠️ 用工具管理长期记忆

记忆操作以**工具**的形式暴露，LLM 可以调用这些工具来存储或检索记忆。

**基于工具的记忆管理：**
- 由 LLM 决定何时存储/检索记忆
- Redis 调用更少，但可能遗漏一些上下文
- 由于 LLM 决策而增加一些延迟

或者，你也可以在工作流中始终手动管理记忆。

**手动记忆管理：**
- Redis 调用更多，但响应时间更快
- 提取更多记忆，提供更丰富的上下文
- 由于上下文更多，token 用量更高

> 注意：**本教程使用基于工具的记忆**，以在控制力和效率之间取得最佳平衡。

<img src="assets/memory-agents.png" width="600" alt="Memory Agents Diagram">

### 定义 Agent 工具

既然已经定义了存储函数，我们就可以创建让 agent 与记忆系统交互的工具了。这些工具将由 LLM 在对话期间用来管理记忆。

我们先从「存储记忆工具」开始：

#### 存储记忆工具（Store Memory Tool）

该工具使 agent 能够把重要信息作为长期记忆保存到 Redis 中。它特别适合捕获：
- 用户偏好和习惯
- 个人经历和趣事
- 对话中分享的重要事实和知识

该工具接受以下参数：
- `content`：要存储的实际记忆内容（例如「用户在航班上喜欢靠窗座位」）
- `memory_type`：记忆类型（例如 `MemoryType.EPISODIC` 表示个人经历，`MemoryType.SEMANTIC` 表示通用知识）
- `metadata`：可选的字典，用于附加上下文（例如时间戳、来源、置信度）
- `config`：可选的用户/thread 上下文配置（由 agent 自动处理）

调用时，该工具会：
1. 校验输入参数
2. 将记忆存储到 Redis 并建立适当索引
3. 返回带有已存储内容的成功消息
4. 用信息丰富的消息优雅地处理错误

该工具旨在供 LLM 使用，构建用户偏好和经历的持久记忆，从而随着时间推移实现更个性化、更具备上下文感知的交互。

In [ ]:
from typing import Dict, Optional

from langchain_core.tools import tool
from langchain_core.runnables.config import RunnableConfig


@tool
def store_memory_tool(
    content: str,
    memory_type: MemoryType,
    metadata: Optional[Dict[str, str]] = None,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Store a long-term memory in the system.

    Use this tool to save important information about user preferences,
    experiences, or general knowledge that might be useful in future
    interactions.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)
    thread_id = config.get("thread_id")

    try:
        # Store in long-term memory
        store_memory(
            content=content,
            memory_type=memory_type,
            user_id=user_id,
            thread_id=thread_id,
            metadata=str(metadata) if metadata else None,
        )

        return f"Successfully stored {memory_type} memory: {content}"
    except Exception as e:
        return f"Error storing memory: {str(e)}"


测试该工具：

In [ ]:
store_memory_tool.invoke({"content": "I like flying on Delta when possible", "memory_type": "episodic"})

既然我们已经了解了如何存储记忆，接下来看看如何检索它们。

#### 检索记忆工具（Retrieve Memories Tool）
该工具让我们能够使用语义相似度和过滤来搜索已存储的记忆。

当你想做以下事情时，该工具尤其有用：
- 查找相关的过往经历或偏好
- 按类型（情景或语义）过滤记忆
- 获取用户特定的信息
- 限制结果数量，让回复保持聚焦

该工具的工作方式：
1. 接收查询字符串并搜索语义相似的记忆
2. 根据记忆类型过滤结果
3. 应用相似度阈值以确保相关性
4. 以清晰、易读的方式格式化结果

In [ ]:
@tool
def retrieve_memories_tool(
    query: str,
    memory_type: List[MemoryType],
    limit: int = 5,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Retrieve long-term memories relevant to the query.

    Use this tool to access previously stored information about user
    preferences, experiences, or general knowledge.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)

    try:
        # Get long-term memories
        stored_memories = retrieve_memories(
            query=query,
            memory_type=memory_type,
            user_id=user_id,
            limit=limit,
            distance_threshold=0.3,
        )

        # Format the response
        response = []

        if stored_memories:
            response.append("Long-term memories:")
            for memory in stored_memories:
                response.append(f"- [{memory.memory_type}] {memory.content}")

        return "\n".join(response) if response else "No relevant memories found."

    except Exception as e:
        return f"Error retrieving memories: {str(e)}"

测试该工具：

In [ ]:
retrieve_memories_tool.invoke({"query": "Airline preferences", "memory_type": ["episodic"]})

# 🌎 构建旅行 Agent

## 配置 ReAct Agent

我们将使用 LangGraph 的预构建组件创建一个具备记忆能力的 ReAct agent：

1. **短期记忆**：checkpoint saver 按 thread 跟踪对话历史
2. **长期记忆**：我们将从对话中提取并存储关键信息
   - 情景记忆（Episodic）：用户偏好和经历
   - 语义记忆（Semantic）：通用旅行知识

系统会自动对对话进行摘要以管理上下文，同时把重要细节保存在长期存储中。

下面我们首先配置 Redis checkpointer（`RedisSaver`），它负责处理 agent 的短期记忆。

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt.chat_agent_executor import create_react_agent
from langgraph.checkpoint.redis import RedisSaver

# Set up the Redis checkpointer for short term memory
redis_saver = RedisSaver(redis_client=redis_client)
redis_saver.setup()

接下来我们为 agent 定义工具集。

In [ ]:
# Define the set of tools
tools = [store_memory_tool, retrieve_memories_tool]

配置来自 OpenAI 的 LLM。

In [ ]:
# Configure an LLM for the agent with a more creative temperature.
llm = ChatOpenAI(model="gpt-4o", temperature=0.7).bind_tools(tools)

将 LLM、工具、checkpointer 和 system prompt 组合起来，装配 ReAct agent！

In [ ]:
# Define the travel agent
travel_agent = create_react_agent(
    model=llm,
    tools=tools,               # Long-term memory: provided as a set of custom tools
    checkpointer=redis_saver,  # Short-term memory: the conversation history
    prompt=SystemMessage(
        content="""
        You are a travel assistant helping users plan their trips. You remember user preferences
        and provide personalized recommendations based on past interactions.

        You have access to the following types of memory:
        1. Short-term memory: The current conversation thread
        2. Long-term memory:
           - Episodic: User preferences and past trip experiences (e.g., "User prefers window seats")
           - Semantic: General knowledge about travel destinations and requirements

        Your procedural knowledge (how to search, book flights, etc.) is built into your tools and prompts.

        Always be helpful, personal, and context-aware in your responses.
        """
    ),
)

✅ 现在基础 agent 已经就位，我们将构建一个 LangGraph 工作流，把这个 agent 作为一个节点来调用。这个图总共由三个节点组成，我们逐个讲解。

## 节点 1：回复用户
在 LangGraph 中，**节点**表示工作流中的一个独立处理单元。每个节点都是一个函数，接收状态对象和配置作为输入，处理数据，并返回更新后的状态。节点可以连接起来，形成定义执行流程的有向图。

`respond_to_user` 节点（如下）是旅行 agent 工作流中的第一个节点。它是用户交互的入口点，负责处理核心对话流程。它的工作方式如下：

1. 接收当前对话状态和配置
2. 从状态中提取所有 human 消息
3. 调用我们的旅行 agent 生成回复
4. 优雅地处理任何错误
5. 用 agent 的回复更新对话状态

该节点使用一个继承自 `MessagesState` 的自定义 `RuntimeState` 类来维护对话历史。这个状态对象在图中的节点之间传递，使每个节点都能按需访问和修改对话上下文。

In [ ]:

from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState


class RuntimeState(MessagesState):
    """Runtime state for the travel agent."""
    pass


def respond_to_user(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    """Invoke the travel agent to generate a response."""
    human_messages = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not human_messages:
        logger.warning("No HumanMessage found in state")
        return state

    try:
        # Single agent invocation, not streamed (simplified for reliability)
        result = travel_agent.invoke({"messages": state["messages"]}, config=config)
        agent_message = result["messages"][-1]
        state["messages"].append(agent_message)
    except Exception as e:
        logger.error(f"Error invoking travel agent: {e}")
        agent_message = AIMessage(
            content="I'm sorry, I encountered an error processing your request."
        )
        state["messages"].append(agent_message)

    return state

## 节点 2：执行工具

`execute_tools` 节点是旅行 agent 工作流中的关键组件，它弥合了 LLM 的决策与实际工具执行之间的鸿沟。它位于 `respond_to_user` 节点之后，负责处理 agent 工具使用能力的实践层面。

当 LLM 判定需要使用某个工具时，它会在回复中包含工具调用。然后该节点会：

1. 扫描对话历史，找到包含工具调用的最新 AI 消息
2. 对找到的每个工具调用：
   - 从消息中提取工具名称、参数和调用 ID
   - 将工具名称与我们的可用工具进行匹配
   - 使用提供的参数执行工具
   - 创建包含结果的 ToolMessage
3. 处理工具执行过程中出现的任何错误
4. 将所有工具结果加回对话历史

这个节点至关重要，因为它使我们的 agent 能够与外部系统和服务交互，同时保持连贯的对话流程。没有它，agent 就只能生成文本回复，而无法执行实际操作或检索实时信息。

In [ ]:
from langchain_core.messages import ToolMessage


def execute_tools(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    """Execute tools specified in the latest AIMessage and append ToolMessages."""
    messages = state["messages"]
    latest_ai_message = next(
        (m for m in reversed(messages) if isinstance(m, AIMessage) and m.tool_calls),
        None
    )

    if not latest_ai_message:
        return state  # No tool calls to process

    tool_messages = []
    for tool_call in latest_ai_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id = tool_call["id"]

        # Find the corresponding tool
        tool = next((t for t in tools if t.name == tool_name), None)
        if not tool:
            continue  # Skip if tool not found

        try:
            # Execute the tool with the provided arguments
            result = tool.invoke(tool_args, config=config)
            # Create a ToolMessage with the result
            tool_message = ToolMessage(
                content=str(result),
                tool_call_id=tool_id,
                name=tool_name
            )
            tool_messages.append(tool_message)
        except Exception as e:
            # Handle tool execution errors
            error_message = ToolMessage(
                content=f"Error executing tool '{tool_name}': {str(e)}",
                tool_call_id=tool_id,
                name=tool_name
            )
            tool_messages.append(error_message)

    # Append the ToolMessages to the message history
    messages.extend(tool_messages)
    state["messages"] = messages
    return state

## 节点 3：对话摘要

对话摘要节点通过把聊天历史浓缩成简洁的摘要来帮助管理上下文。这可以防止 LLM 的上下文窗口随着对话增长而不堪重负。

关键特性：

1. **自动触发**：每 6 条消息后摘要一次（可配置）

2. **智能摘要**：
   - 使用低 temperature（0.3）的 GPT-4o 以生成一致的摘要
   - 保留偏好和待处理事项等关键信息
   - 替换旧消息，同时保留近期上下文

3. **优点**：
   - 防止上下文窗口溢出
   - 保持对话连贯性
   - 优化 token 用量

摘要会成为对话历史的一部分，让 agent 能够高效地引用过往交互。

In [ ]:
from langchain_core.messages import RemoveMessage

# An LLM configured for summarization.
summarizer = ChatOpenAI(model="gpt-4o", temperature=0.3)

# The number of messages after which we'll summarize the conversation.
MESSAGE_SUMMARIZATION_THRESHOLD = 6


def summarize_conversation(
    state: RuntimeState, config: RunnableConfig
) -> RuntimeState:
    """
    Summarize a list of messages into a concise summary to reduce context length
    while preserving important information.
    """
    messages = state["messages"]
    current_message_count = len(messages)
    if current_message_count < MESSAGE_SUMMARIZATION_THRESHOLD:
        logger.debug(f"Not summarizing conversation: {current_message_count}")
        return state

    system_prompt = """
    You are a conversation summarizer. Create a concise summary of the previous
    conversation between a user and a travel assistant.

    The summary should:
    1. Highlight key topics, preferences, and decisions
    2. Include any specific trip details (destinations, dates, preferences)
    3. Note any outstanding questions or topics that need follow-up
    4. Be concise but informative

    Format your summary as a brief narrative paragraph.
    """

    message_content = "\n".join(
        [
            f"{'User' if isinstance(msg, HumanMessage) else 'Assistant'}: {msg.content}"
            for msg in messages
        ]
    )

    # Invoke the summarizer
    summary_messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=f"Please summarize this conversation:\n\n{message_content}"
        ),
    ]

    summary_response = summarizer.invoke(summary_messages)

    logger.info(f"Summarized {len(messages)} messages into a conversation summary")

    summary_message = SystemMessage(
        content=f"""
        Summary of the conversation so far:

        {summary_response.content}

        Please continue the conversation based on this summary and the recent messages.
        """
    )
    remove_messages = [
        RemoveMessage(id=msg.id) for msg in messages if msg.id is not None
    ]

    state["messages"] = [  # type: ignore
        *remove_messages,
        summary_message,
        state["messages"][-1],
    ]

    return state.copy()

## 装配完整图

🚧 现在是时候装配我们的图，以实现端到端的 agent 执行了。我们会把上面定义的三个**节点**都接进来。

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(RuntimeState)

# Add nodes to the graph
workflow.add_node("agent", respond_to_user)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("summarize_conversation", summarize_conversation)

接下来，我们用控制流程走向的**边（edges）**把节点连接起来。agent 节点与后续节点之间有一条条件边，后续走向取决于我们是需要处理并执行工具调用，还是继续前进……

In [ ]:
def decide_next_step(state):
    latest_ai_message = next((m for m in reversed(state["messages"]) if isinstance(m, AIMessage)), None)
    if latest_ai_message and latest_ai_message.tool_calls:
        return "execute_tools"
    return "summarize_conversation"


workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    decide_next_step,
    {"execute_tools": "execute_tools", "summarize_conversation": "summarize_conversation"},
)
workflow.add_edge("execute_tools", "agent")
workflow.add_edge("summarize_conversation", END)

编译这个图！

In [ ]:
graph = workflow.compile(checkpointer=redis_saver)

## 测试主 Agent 循环

让我们让旅行 agent 开始工作吧！主循环负责处理对话流程：

**初始化**：设置唯一的 thread ID 和空的消息状态

**输入处理**：获取用户输入，处理空输入和退出命令

**消息流转**：将输入转换为 HumanMessage 并通过我们的工作流进行流式处理

**回复生成**：处理状态并显示 AI 回复

**错误处理**：捕获错误并让对话流畅地继续

In [ ]:
def main(thread_id: str = "book_flight", user_id: str = "demo_user"):
    """Main interaction loop for the travel agent"""

    print("Welcome to the Travel Assistant! (Type 'exit' to quit)")

    config = RunnableConfig(configurable={"thread_id": thread_id, "user_id": user_id})
    state = RuntimeState(messages=[])

    while True:
        user_input = input("\nYou (type 'quit' to quit): ")

        if not user_input:
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("Thank you for using the Travel Assistant. Goodbye!")
            break

        state["messages"].append(HumanMessage(content=user_input))

        try:
            # Process user input through the graph
            for result in graph.stream(state, config=config, stream_mode="values"):
                state = RuntimeState(**result)

            logger.debug(f"# of messages after run: {len(state['messages'])}")

            # Find the most recent AI message, so we can print the response
            ai_messages = [m for m in state["messages"] if isinstance(m, AIMessage)]
            if ai_messages:
                message = ai_messages[-1].content
            else:
                logger.error("No AI messages after run")
                message = "I'm sorry, I couldn't process your request properly."
                # Add the error message to the state
                state["messages"].append(AIMessage(content=message))

            print(f"\nAssistant: {message}")

        except Exception as e:
            logger.exception(f"Error processing request: {e}")
            error_message = "I'm sorry, I encountered an error processing your request."
            print(f"\nAssistant: {error_message}")
            # Add the error message to the state
            state["messages"].append(AIMessage(content=error_message))

在你亲自尝试之前，先看一下 Tyler 和旅行 agent 之间的这段对话。注意工作流过程中发生的记忆存储操作、对 LLM 的调用，以及对话摘要！

In [ ]:
try:
    user_id = input("Enter a user ID: ") or "demo_user"
    thread_id = input("Enter a thread ID: ") or "demo_thread"
except Exception:
    # If we're running in CI, we don't have a terminal to input from, so just exit
    exit()
else:
    main(thread_id, user_id)


让我们回顾一下 agent 在这个过程中学到了关于我的哪些信息！

In [ ]:
res = retrieve_memories_tool.invoke({"query": "Travel, activity, and dietary preferences", "memory_type": ["episodic", "semantic"]})
res.split("\n")

别忘了，我们还有 RedisVL 索引，可以按需手动查询或使用：

In [ ]:
from redisvl.query import CountQuery

# count total long-term memories in Redis
long_term_memory_index.query(CountQuery())

___


# 🎓 回顾

你现在已经从头学习了基本原理，并从零构建了一个**生产级、具备记忆能力的 AI agent**。让我们回顾一下关键成果：

## 🏗️ 我们构建了什么

1. ✅ **双记忆架构**：使用 LangGraph 和 Redis 实现短期对话状态 + 长期持久化知识
2. ✅ **向量驱动的记忆**：使用 RedisVL 进行语义搜索
3. ✅ **智能去重**：避免多次存储相似记忆
4. ✅ **基于工具的记忆管理**：由 LLM 控制何时存储/检索记忆
5. ✅ **对话摘要**：自动管理上下文窗口

**为什么选择 Redis？**

- **性能**：大规模场景下亚毫秒级的记忆检索
- **多面性**：既能处理结构化状态（checkpoints），也能处理非结构化数据（vectors）
- **生产就绪**：内置持久化、集群和高可用
- **开发者体验**：拥有 RedisVL 等工具以及 AI 框架集成的丰富生态

## 🔧 替代的记忆开发框架

虽然本教程展示的是动手实现，但你也可以考虑以下框架来加快开发速度：

- **[LangMem](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=github-langmem&target=https%3A%2F%2Fgithub.com%2Flangchain-ai%2Flangmem&text=LangMem)**：LangChain 官方的记忆框架
- **[Mem0](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=github-mem0&target=https%3A%2F%2Fgithub.com%2Fmem0-ai%2Fmem0&text=Mem0)**：面向 AI 应用的专用记忆层

**何时使用哪种方案：**
- **自定义实现**（本教程）：最大控制力、特定需求、学习目的
- **LangMem**：LangChain 生态集成、快速原型
- **Mem0**：多应用记忆共享、企业级特性

## 🔄 下一步

你现在已经具备了构建复杂、具备记忆能力的 AI agent 的基础，这些 agent 会显得真正智能和个性化。

**需要专家指导？**
#
[预约咨询](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-redis--agent-memory-tutorial&click=redis-meeting&target=https%3A%2F%2Fredis.io%2Fmeeting%2F%3Futm_source%3Dnir%26utm_medium%3Dcpa%26utm_campaign%3D2025-06-ai_in_production%26utm_content%3Dcr-all_contact_us_forms-7013z000000evRk&text=Schedule%20a%20consultation)，与我们的 Redis 专家一起优化你的 agent 架构和实现。